In [2]:
import pandas as pd
import ast

path = r"C:\Users\fonse\SLIIT\projects\Research-Project\Component 2\data\SOLD_train.csv"

# ✅ IMPORTANT: SOLD file is tab-separated
df = pd.read_csv(path, sep="\t", engine="python", encoding="utf-8")

# Convert rationales from string to list safely
def parse_rationales(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    # handles "[]" and "[0, 1, 0]"
    return ast.literal_eval(s)

df["rationales_list"] = df["rationales"].apply(parse_rationales)

# Function to extract hate words
def extract_hate_words(row):
    tokens = str(row["tokens"]).split()
    r = row["rationales_list"]
    n = min(len(tokens), len(r))
    hate_words = [tokens[i] for i in range(n) if int(r[i]) == 1]
    return ", ".join(hate_words)   # comma separated (or return hate_words as list)

# Create new column with hate words
df["hate_words"] = df.apply(extract_hate_words, axis=1)

# Create binary hate column (1 if label is OFF OR if hate_words exists)
df["hate"] = df["label"].astype(str).str.upper().eq("OFF").astype(int)
# (alternative) df["hate"] = df["hate_words"].apply(lambda x: 1 if x.strip() else 0)

# Preview
df[["tokens", "rationales", "label", "hate_words", "hate"]].head()

,tokens,rationales,label,hate_words,hate
0,@USER @USER පට්ට පට පට . . .,[],NOT,,0
1,පරණ කෑල්ල අද වෙනකම් හිටියනම් අදට අවුරුදු 4යි ....,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",OFF,ටෞකනවා,1
2,යාළුවා කියලා හිතන් සර් ගේ ඔලුවට රෙද්ද දාලා නෙල...,[],NOT,,0
3,හොඳ මිතුරියක් කතා කලා . විස්තර කතාකරමින් ඉදලා ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",OFF,"තෝ, ගෑනියෙක්, බිජ්ජට",1
4,"ඔය බනින්නෙ . . හරකා , මී හරකා කිය කිය . . .","[0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0]",OFF,"හරකා, මී, හරකා",1


In [3]:
df.to_csv("SOLD_train_with_hate_column.csv", index=False)

In [14]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

Current folder: C:\Users\fonse\SLIIT\projects\Research-Project\Component 2\notebooks
Files here: ['.ipynb_checkpoints', 'SOLD_train_with_hate_column.csv', 'Untitled.ipynb']


In [18]:
import os
import math
import pandas as pd

df = pd.read_csv(r"C:\Users\fonse\SLIIT\projects\Research-Project\Component 2\SOLD_train_with_hate_column.csv")
# Example: df.to_csv("SOLD_train_with_hate_column.csv", index=False)

batch_size = 200
out_dir = "batches_200"
os.makedirs(out_dir, exist_ok=True)

num_batches = math.ceil(len(df) / batch_size)

for i in range(num_batches):
    start = i * batch_size
    end = start + batch_size
    batch_df = df.iloc[start:end].copy()
    batch_df.to_csv(os.path.join(out_dir, f"batch_{i+1}.csv"), index=False)

print(f"✅ Done! Created {num_batches} files inside folder: {out_dir}")

✅ Done! Created 38 files inside folder: batches_200
